# Sesión 06 — Random Forest, Gradient Boosting y boosting moderno

**Bloque:** 2 · Regresión

## Objetivos

Al terminar esta sesión deberías ser capaz de:

- Explicar en qué se diferencian *bagging* y *boosting* sin recurrir a la fórmula: qué hace cada uno con los árboles y por qué funciona.
- Justificar por qué promediar muchos árboles inestables produce un modelo estable.
- Ajustar `n_estimators` y `learning_rate` entendiendo que están acoplados, y predecir qué pasa si mueves uno solo.
- Comparar modelos por **rendimiento y por coste** —tiempo de entrenamiento— y defender una elección que tenga en cuenta los dos.
- Usar XGBoost, LightGBM y CatBoost, y saber decir qué aporta cada uno frente al `GradientBoostingRegressor` de scikit-learn.
- Reconocer cuándo merece la pena tratar las variables categóricas de forma nativa en vez de con `get_dummies`.

## Qué necesitas de antes

- **S05**: árboles de regresión, `max_depth`, sobreajuste y `GridSearchCV`. Un ensemble no es más que muchos árboles de S05 puestos de acuerdo.
- **S04**: validación cruzada.
- Tus RMSE de S04 y S05 sobre **California** y sobre **Ames**. Hoy se cierra la tabla del bloque entero.

## Librerías

Además del stack de siempre, esta sesión usa **`xgboost`**, **`lightgbm`** y **`catboost`**. Si no los tienes:

```
pip install xgboost lightgbm catboost
```

## 1. Random Forest: muchos árboles mediocres, una buena decisión

En S05 quedó una pega: un árbol es **inestable**. Cambia unas pocas filas del entrenamiento y el árbol se reorganiza entero. Un modelo así acierta o falla según qué datos le tocaron, que es justo lo que no queremos.

El **Random Forest** ataca esa inestabilidad de la forma más simple posible: en vez de un árbol, entrena **muchos** y promedia sus predicciones. La estrategia se llama ***bagging*** (*bootstrap aggregating*) y tiene dos ingredientes:

1. **Cada árbol ve un remuestreo distinto** de los datos de entrenamiento (filas sacadas al azar con reemplazo).
2. **Cada corte considera solo un subconjunto al azar de las variables** (`max_features`), para que los árboles no se parezcan demasiado entre sí.

La clave está en el promedio. Cada árbol individual se equivoca, pero **se equivoca en una dirección distinta**, porque ha visto datos distintos. Al promediar, los errores aleatorios se cancelan y queda la señal común. Por eso funciona: no porque cada árbol sea bueno, sino porque sus errores son independientes.

Hiperparámetros que importan:

- **`n_estimators`** — cuántos árboles. **Más nunca empeora**; solo cuesta tiempo, con rendimientos decrecientes. Esto es específico de *bagging* y no vale para *boosting*.
- **`max_depth`, `min_samples_leaf`** — los de S05, aplicados a cada árbol del bosque.
- **`max_features`** — cuántas variables puede mirar cada corte. Bajarlo hace los árboles más distintos entre sí, que es de donde sale la ganancia.

**Qué problema resuelve:** la varianza. Un modelo que depende demasiado de qué filas concretas vio.

**Cuándo elegirlo:** como primer modelo serio en casi cualquier problema tabular. Da un resultado muy digno sin tocar nada, es difícil de estropear y se paraleliza sin esfuerzo, porque los árboles son independientes.

**Cuándo NO:** cuando necesitas explicar la decisión —ya no hay un árbol que enseñar—, cuando la predicción tiene que ser muy rápida, o cuando el problema es de verdad lineal.

**Qué asume:** lo mismo que un árbol, es decir, casi nada. Hereda también sus límites: no extrapola fuera del rango visto.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")

### Demostración: el bosque contra el árbol

Un dataset de juguete: una curva seno con ruido añadido. Sabemos cuál es la señal verdadera, así que podemos ver quién la encuentra y quién persigue el ruido.

In [ ]:
# Datos de juguete: una curva seno con ruido en uno de cada cinco puntos
rng = np.random.RandomState(42)
X_juguete = np.sort(5 * rng.rand(80, 1), axis=0)
y_juguete = np.sin(X_juguete).ravel()
y_juguete[::5] += 3 * (0.5 - rng.rand(16))

# Un árbol solo, con margen de sobra para sobreajustar
arbol = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_juguete, y_juguete)

# Un bosque de 100 árboles sobre los mismos datos
bosque = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_juguete, y_juguete)

rejilla = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]

plt.figure(figsize=(12, 7))
plt.scatter(X_juguete, y_juguete, s=20, edgecolor="black", c="cornflowerblue",
            label="Datos con ruido")
plt.plot(rejilla, np.sin(rejilla), color="gray", linestyle="--", lw=2,
         label="Señal verdadera (sin ruido)")
plt.plot(rejilla, arbol.predict(rejilla), color="red", lw=2,
         label="Un solo árbol (max_depth=5)")
plt.plot(rejilla, bosque.predict(rejilla), color="darkgreen", lw=2,
         label="Random Forest (100 árboles)")
plt.xlabel("Variable de entrada")
plt.ylabel("Objetivo")
plt.title("Un árbol contra un bosque, sobre los mismos datos", fontsize=16)
plt.legend()
plt.show()

**Qué se ve.** La línea gris discontinua es la señal verdadera, el seno limpio: es lo que un modelo perfecto debería encontrar.

La **roja** es el árbol único. Se lanza a por cada punto, incluidos los que solo son ruido, y produce esa silueta de escalones bruscos. Ha confundido el ruido con información.

La **verde** es el bosque. Está claramente más cerca del seno verdadero. Y lo notable es que **cada uno de sus 100 árboles es tan malo como el rojo**: ninguno mejoró. Lo que mejora es el promedio, porque cada árbol vio un remuestreo distinto y por tanto se equivocó en un sitio distinto. Al sumarlos, los errores se compensan y sobrevive lo que todos tienen en común, que es la señal.

Fíjate en que la verde sigue siendo escalonada, no suave: sigue siendo una suma de árboles, con sus límites. Y en los extremos ninguna de las dos extrapola.

---
### Preguntas de recap — Random Forest

1. ¿Qué problema de los árboles individuales resuelve el Random Forest?
2. ¿Por qué hace falta que cada árbol vea datos distintos? ¿Qué pasaría si todos vieran lo mismo?
3. ¿Qué controla `max_features` y por qué bajarlo puede mejorar el modelo?
4. ¿Puedes subir `n_estimators` sin miedo a sobreajustar? ¿Qué te cuesta subirlo?
5. En la demostración, ningún árbol del bosque es mejor que el árbol rojo. ¿Cómo puede el conjunto ser mejor que cualquiera de sus partes?

---

## 2. Gradient Boosting: cada árbol arregla lo que rompió el anterior

El **Gradient Boosting** llega al mismo sitio por el camino contrario. En vez de entrenar árboles independientes y promediarlos, los entrena **en cadena**: cada árbol nuevo se dedica a predecir **los errores que aún quedan** del conjunto anterior.

El procedimiento, sin fórmulas:

1. Empieza con una predicción tonta (por ejemplo, la media de `y`).
2. Calcula en qué se ha equivocado: los **residuos**.
3. Entrena un árbol pequeño **para predecir esos residuos**.
4. Suma ese árbol a la predicción, multiplicado por un factor pequeño.
5. Vuelve al paso 2, ahora con los errores que quedan.

De ahí salen las dos diferencias prácticas frente a Random Forest:

- Los árboles de boosting son **deliberadamente débiles**: profundidad 3 o menos. No hacen falta buenos, hacen falta muchos y especializados.
- **`n_estimators` sí puede sobreajustar.** En *bagging* añadir árboles solo afina el promedio; aquí cada árbol añade una corrección, y si sigues corrigiendo acabas corrigiendo el ruido.

El hiperparámetro que manda es el **`learning_rate`**: el factor por el que se multiplica cada árbol antes de sumarlo. Cuanto más bajo, más pequeño el paso, más robusto el modelo... **y más árboles hacen falta para llegar**. `learning_rate` y `n_estimators` están acoplados: mover uno sin mover el otro suele empeorar las cosas. Lo comprobarás en el ejercicio.

**Qué problema resuelve:** el sesgo. Va reduciendo el error sistemático que queda, corrección a corrección.

**Cuándo elegirlo:** cuando quieres el mejor rendimiento posible en datos tabulares y estás dispuesto a ajustar. En la práctica, es lo que gana las competiciones de datos tabulares.

**Cuándo NO:** cuando no vas a ajustar nada (Random Forest da más por menos), cuando el entrenamiento tiene que ser rápido y no puedes usar las librerías modernas, o cuando hay mucho ruido en la variable objetivo — perseguir residuos ruidosos es perseguir ruido.

**Qué asume:** como el árbol, casi nada. Pero es bastante más sensible a sus hiperparámetros, y por tanto más fácil de estropear.

In [ ]:
# Boosting a mano, con dos árboles, sobre los mismos datos de juguete

# Árbol 1: deliberadamente débil (una sola pregunta)
arbol_1 = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X_juguete, y_juguete)
pred_1 = arbol_1.predict(X_juguete)

# Lo que le queda por explicar: los residuos
residuos_1 = y_juguete - pred_1

# Árbol 2: NO predice y, predice el error del árbol 1
arbol_2 = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X_juguete, residuos_1)
pred_2 = arbol_2.predict(X_juguete)

# La predicción combinada es la suma de las dos
pred_combinada = pred_1 + pred_2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.scatter(X_juguete, y_juguete, s=20, edgecolor="black", c="cornflowerblue",
            label="Datos reales")
ax1.plot(X_juguete, pred_1, color="red", lw=2, label="Predicción del árbol 1")
ax1.scatter(X_juguete, residuos_1, s=20, edgecolor="black", c="orange",
            label="Residuos: lo que falla el árbol 1")
ax1.axhline(0, color='gray', linestyle='--')
ax1.set_title("Paso 1: un árbol débil y lo que le sobra", fontsize=15)
ax1.legend()

ax2.scatter(X_juguete, y_juguete, s=20, edgecolor="black", c="cornflowerblue",
            label="Datos reales")
ax2.plot(X_juguete, pred_1, color="red", lw=1, alpha=0.5, label="Solo el árbol 1")
ax2.plot(X_juguete, pred_combinada, color="darkgreen", lw=2,
         label="Árbol 1 + árbol 2")
ax2.set_title("Paso 2: sumamos la corrección", fontsize=15)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"RMSE solo con el árbol 1:      {np.sqrt(np.mean(residuos_1 ** 2)):.3f}")
print(f"RMSE con los dos combinados:   {np.sqrt(np.mean((y_juguete - pred_combinada) ** 2)):.3f}")

**Qué se ve.**

En el gráfico de la izquierda, la línea roja es un árbol de profundidad 1: hace **una sola pregunta** y por tanto solo puede dar dos valores. Es malísimo a propósito. Los puntos naranjas son sus residuos, y lo importante es que **no son ruido aleatorio**: tienen forma. Esa forma es información que el árbol 1 no supo aprovechar y que, por tanto, todavía está disponible.

A la derecha, el árbol 2 —entrenado para predecir esos naranjas, no para predecir `y`— se suma al primero. La línea verde tiene ya cuatro escalones en vez de dos, y el RMSE baja.

Repite eso cien veces y tienes Gradient Boosting. Cada árbol es ridículo por separado; la cadena entera no lo es.

Dos matices sobre esta demostración simplificada: el modelo real multiplica cada árbol por el `learning_rate` antes de sumarlo, en vez de sumarlo entero como aquí; y arranca de la media de `y`, no del primer árbol. La idea es la que has visto.

---
### Preguntas de recap — Gradient Boosting

1. ¿En qué se diferencia el Gradient Boosting del Random Forest, en una frase?
2. ¿Qué predice exactamente el segundo árbol de un Gradient Boosting?
3. ¿Por qué los árboles de boosting se dejan a propósito poco profundos?
4. ¿Qué hace el `learning_rate` y por qué no se puede tocar sin mirar `n_estimators`?
5. ¿Por qué en boosting sí puede sobreajustar subir `n_estimators`, si en Random Forest no?

---

---
## Ejercicio 1 — La gran comparativa

**Contexto.** Sigues en la inmobiliaria de California. Ya les entregaste un árbol ajustado en S05 y quedaron satisfechos. Ahora te preguntan si se puede hacer mejor, y el director de tecnología añade una condición: *"lo que sea, pero el modelo se reentrena cada noche y tiene que estar listo antes de que abramos"*. Es decir: **el tiempo de entrenamiento entra en la decisión**.

**Tu tarea.**
1. Prepara California como en S05: numéricas, nulos a la mediana, `train_test_split` con `random_state=42`. Los árboles no necesitan escalado.
2. Entrena `DecisionTreeRegressor`, `RandomForestRegressor` y `GradientBoostingRegressor` **con los parámetros por defecto** y `random_state=42`. Para cada uno mide **el RMSE en test y el tiempo de entrenamiento**.
3. Monta una tabla de pandas con las tres filas y las dos columnas.
4. Saca las `feature_importances_` de los tres y dibújalas en un gráfico de barras agrupadas, para ver si coinciden en qué importa.
5. Entrena otro `GradientBoostingRegressor` cambiando solo `learning_rate` a `0.01`. Compara su RMSE con el del modelo por defecto. Después prueba a subir `n_estimators` a 1000 manteniendo ese `learning_rate`, y mide también el tiempo.

**Criterio de que lo has hecho bien:** los tres modelos deben quedar claramente ordenados en RMSE, y **el orden por tiempo no es el mismo que el orden por acierto** — ahí está la decisión que te piden. En el punto 5, bajar el `learning_rate` a secas debería **empeorar mucho** las cosas; si te mejora, revisa el código. Al subir `n_estimators` deberías recuperar aproximadamente lo perdido, a un coste de tiempo que se nota.

In [ ]:
# TODO: completa esta celda
# Para el tiempo te basta con `time.time()` antes y después del `fit`; no cronometres la predicción.
# Guarda los modelos ya entrenados en un diccionario: los vas a necesitar en el apartado de importancias.


In [ ]:
# TODO: completa esta celda
# Para el gráfico agrupado, un DataFrame con una columna por modelo y las variables en el índice se dibuja solo.
# En el punto 5 no cambies nada más que lo que te piden: si tocas dos cosas a la vez no sabrás cuál causó qué.


### Mis reflexiones

*(Escribe aquí tu respuesta antes de seguir)*

- El modelo que más acierta es el que más tarda, y el que menos acierta es instantáneo. Con la condición que puso el director de tecnología, ¿cuál entregas? Escribe el argumento en dos frases, como se lo dirías a él.
- Random Forest le ha ganado a Gradient Boosting. ¿Concluyes que promediar es mejor que corregir errores? Piensa qué es exactamente lo que has comparado antes de responder.
- Bajaste el `learning_rate` esperando un modelo más robusto y salió mucho peor. Explica qué pasó, y qué habrías tenido que hacer a la vez.
- Los tres modelos coinciden en cuál es la variable más importante. ¿Te da eso más confianza en la conclusión que si lo dijera uno solo? ¿Por qué?

## 3. Boosting moderno: XGBoost, LightGBM y CatBoost

El `GradientBoostingRegressor` de scikit-learn implementa la idea original de 2001 de forma honesta y directa. Desde entonces han aparecido tres librerías que hacen lo mismo **mucho más rápido y algo mejor**, y que son lo que se usa hoy en la práctica. No son algoritmos distintos: son la misma idea con la ingeniería resuelta.

**De dónde sale la velocidad.** El cuello de botella de un árbol es buscar el mejor corte: hay que probar cada valor de cada variable. Las tres librerías lo evitan agrupando cada variable en un centenar de **cubos** (*histogramas*) y probando solo los bordes de los cubos. Se pasa de miles de cortes candidatos a unos cien, con una pérdida de precisión mínima. Súmale paralelismo y uso cuidadoso de la memoria y salen los órdenes de magnitud que vas a medir.

**Qué aporta cada una:**

- **XGBoost** — la que popularizó todo esto. Muy robusta, con regularización incorporada (`reg_alpha`, `reg_lambda` — sí, los de S04) y la más extendida en producción. Si dudas, esta.
- **LightGBM** — la más rápida, con diferencia. Hace crecer el árbol *por hojas* en vez de por niveles: profundiza donde más ganancia hay en lugar de expandir uniformemente. Muy buena con datasets grandes; con pocos datos puede sobreajustar precisamente por eso.
- **CatBoost** — la que trata **las variables categóricas de forma nativa**. No necesitas `get_dummies`: le pasas qué columnas son categóricas y se encarga. Es la que mejor funciona sin tocar nada, a cambio de entrenar algo más lento.

**Y una cuarta que ya tienes instalada:** `HistGradientBoostingRegressor`, en scikit-learn desde la 0.21. Usa la misma técnica de histogramas y no requiere instalar nada. Para muchos problemas es suficiente.

**Cuándo elegirlo:** siempre que ibas a usar `GradientBoostingRegressor`. No hay ninguna razón práctica para preferir el original salvo no poder instalar dependencias.

**Cuándo NO:** cuando el dataset es pequeño (unos cientos de filas) y un Random Forest te da lo mismo sin ajustar nada; o cuando necesitas explicar el modelo, porque aquí ya no hay nada que dibujar.

**Qué asume:** lo mismo que cualquier árbol. Ojo con una diferencia de forma: cada librería tiene su propio nombre para los mismos parámetros —`n_estimators` frente a `num_boost_round`, `learning_rate` frente a `eta`— aunque las tres ofrecen una interfaz compatible con scikit-learn, que es la que usamos aquí.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# California, preparada igual que en S05 y en el ejercicio 1. La cargamos aquí
# para que la demo funcione aunque el ejercicio 1 esté a medias.
url = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
df_num = pd.read_csv(url).select_dtypes(include=np.number)
df_num = df_num.fillna(df_num.median())
X = df_num.drop('median_house_value', axis=1)
y = df_num['median_house_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# El contraste, en cuatro líneas: lo mismo que hace GradientBoosting, pero moderno
inicio = time.time()
gb_clasico = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
t_clasico = time.time() - inicio
rmse_clasico = np.sqrt(mean_squared_error(y_test, gb_clasico.predict(X_test)))

inicio = time.time()
lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1).fit(X_train, y_train)
t_lgbm = time.time() - inicio
rmse_lgbm = np.sqrt(mean_squared_error(y_test, lgbm.predict(X_test)))

print(f"GradientBoostingRegressor : RMSE {rmse_clasico:>9,.0f}   {t_clasico:>6.2f}s")
print(f"LightGBM                  : RMSE {rmse_lgbm:>9,.0f}   {t_lgbm:>6.2f}s")
print(f"\nLightGBM es {t_clasico / t_lgbm:.0f} veces más rápido "
      f"y además acierta {rmse_clasico - rmse_lgbm:,.0f} $ más.")

Ese es el argumento entero de la segunda mitad de la sesión, y no necesita más justificación: **el mismo trabajo, decenas de veces más rápido, y encima con menos error.**

La velocidad no es un lujo. Es lo que decide si puedes permitirte un `GridSearchCV` de 500 combinaciones o solo de 20; y por tanto, indirectamente, decide cuánto acierta tu modelo final. Lo mides en el ejercicio 2.

---
### Preguntas de recap — Boosting moderno

1. ¿Qué truco usan las tres librerías para ir tan rápido, y qué se pierde a cambio?
2. ¿Qué diferencia a LightGBM de las otras dos en la forma de hacer crecer el árbol?
3. ¿Qué aporta CatBoost que no aportan las otras?
4. ¿Qué usarías si no puedes instalar nada fuera de scikit-learn?
5. La velocidad no solo ahorra tiempo: también mejora el modelo final. ¿Por qué?

---

---
## Ejercicio 2 — Cerrar la tabla del bloque

**Contexto.** Última reunión con la inmobiliaria de California. Llevas cuatro sesiones sobre el mismo dataset y toca presentar el recorrido completo: de dónde partiste, adónde has llegado y qué recomiendas. El director de tecnología sigue con lo suyo: el reentrenamiento nocturno.

**Tu tarea.**
1. Entrena sobre California, con los parámetros por defecto: `HistGradientBoostingRegressor`, `XGBRegressor`, `LGBMRegressor` y `CatBoostRegressor`. Mide RMSE y tiempo de cada uno. Usa `verbose=-1` en LightGBM y `verbose=0` en CatBoost para que no llenen la salida.
2. Monta la **tabla acumulada del bloque 2**, con todos los modelos que has entrenado sobre California a lo largo del curso: tu Lasso de S04, tu árbol tuneado de S05, y los de hoy. Ordénala por RMSE y añade una columna con la mejora porcentual respecto al Lasso.
3. Lanza un `GridSearchCV` sobre `GradientBoostingRegressor` con la rejilla `n_estimators: [100, 200]`, `learning_rate: [0.1, 0.05]`, `max_depth: [3, 5]` y `cv=3`. Cronométralo.
4. Lanza **el mismo `GridSearchCV` sobre LightGBM**, con la rejilla idéntica. Cronométralo también y compara los dos tiempos y los dos RMSE.
5. Con todo eso, escribe tu recomendación en la celda de reflexión.

**Criterio de que lo has hecho bien:** los cuatro modelos modernos deben quedar agrupados y por delante de todo lo anterior, con diferencias pequeñas entre ellos. La tabla acumulada tiene que enseñar una mejora grande —de más de un tercio— desde el primer modelo del bloque. Y en los puntos 3 y 4, la misma búsqueda sobre la misma rejilla debe tardar tiempos que no se parecen en nada.

In [ ]:
# TODO: completa esta celda
# Un diccionario de nombre -> modelo y un bucle te evita repetir el mismo bloque cuatro veces.
# CatBoost y LightGBM son habladores: mira los parámetros que silencian su salida antes de lanzarlos.


In [ ]:
# TODO: completa esta celda
# No reentrenes nada: las listas de resultados del ejercicio y del apartado anterior ya tienen lo que necesitas.
# La mejora porcentual se calcula siempre contra el mismo punto de partida, el Lasso.


In [ ]:
# TODO: completa esta celda
# La rejilla tiene que ser idéntica en las dos búsquedas o la comparación de tiempos no significa nada.
# Recuerda de S05 que el número de ajustes no es el de combinaciones: hay que multiplicar por los folds.


### Mis reflexiones

*(Escribe aquí tu respuesta)*

- Escribe tu recomendación final para la inmobiliaria: qué modelo, y por qué ese y no el que más acierta. Tiene que aparecer el tiempo de entrenamiento en tu argumento.
- Has mejorado un 35 % desde el Lasso de S04. ¿Qué has perdido por el camino? Nombra algo concreto que el Lasso te daba y el CatBoost no.
- La misma búsqueda tarda unos segundos con LightGBM y bastantes más con `GradientBoostingRegressor`. Explica por qué eso no es solo una cuestión de comodidad, sino que acaba afectando a cuánto acierta tu modelo.
- Los cuatro modelos modernos quedan casi empatados. Si tuvieras que elegir uno para todos tus proyectos futuros sin conocerlos de antemano, ¿cuál y por qué?
- El `GradientBoosting` tuneado adelanta al Random Forest por defecto, pero el Random Forest por defecto ganaba al `GradientBoosting` por defecto. ¿Qué te enseña eso sobre las comparativas de modelos que leas por ahí?

---
## Ejercicio 3 — Volver a Ames a rectificar

**Contexto.** En S05 comparaste un árbol contra los modelos lineales sobre Ames y el árbol **perdió**: ~37.570 $ contra los ~36.910 $ de la regresión lineal. La conclusión que escribiste fue que en Ames la relación es esencialmente lineal y que el troceado del árbol no encajaba.

Vamos a comprobar si esa conclusión aguanta ahora que tienes ensembles. Y de paso, a recuperar algo que llevamos tirando a la basura desde S04: **Ames tiene 43 columnas de texto** —barrio, calidad de la cocina, tipo de cimentación— que hemos venido descartando con `select_dtypes(include=np.number)` en todas las sesiones.

**Tu tarea.**

*Parte 1 — ¿Aguanta la conclusión de S05?*
1. Prepara Ames con las columnas numéricas, como en S04 y S05.
2. Entrena `RandomForest`, `GradientBoosting`, `XGBoost`, `LightGBM` y `CatBoost` por defecto, y mide su RMSE en test.
3. Ponlos en una tabla junto a tus RMSE de S04 (`LinearRegression`, Lasso, Ridge) y de S05 (el mejor árbol).

*Parte 2 — Las categóricas que tirábamos*
4. Prepara Ames **entero**, sin descartar las columnas de texto: rellena los nulos de las categóricas con la cadena `"ausente"` y los de las numéricas con la mediana.
5. Entrena un `CatBoostRegressor` pasándole las posiciones de las columnas categóricas en `cat_features`. Tarda unos 45 segundos: es normal.
6. Compáralo con el **CatBoost de la Parte 1**, el que solo veía las numéricas. Esa es la comparación limpia: mismo algoritmo, única diferencia las categóricas.
7. Mira las importancias del modelo completo y cuenta cuántas de las 15 primeras son categóricas.

**Criterio de que lo has hecho bien:** en la Parte 1 los ensembles deberían dar la vuelta al resultado de S05 de forma clara, no por poco. En la Parte 2, añadir las categóricas debería mejorar todavía más. Si alguna de las dos cosas no pasa, revisa que comparas RMSE de test contra RMSE de test.

**Entrega:** el notebook con las celdas completadas y las reflexiones escritas.

In [ ]:
# TODO: completa esta celda
# La preparación es exactamente la de S05: puedes copiarla tal cual.
# Guarda aparte el RMSE del CatBoost numérico: lo necesitas en la Parte 2 para la comparación limpia.


In [ ]:
# TODO: completa esta celda
# `cat_features` espera las posiciones de las columnas, no sus nombres: `columnas.get_loc(...)` te las da.
# CatBoost no admite nulos en las categóricas, por eso hay que rellenarlas con un texto antes de entrenar.


### Mis reflexiones finales

*(Escribe aquí tu respuesta)*

- En S05 escribiste que en Ames ganaba el modelo lineal porque la relación era lineal. Hoy cinco ensembles le sacan un 25 %. Reescribe aquella conclusión: ¿qué le pasaba de verdad al árbol único?
- Generaliza lo anterior. Cuando leas o escribas "el modelo X no funciona para este problema", ¿qué deberías comprobar antes de darlo por bueno?
- Aprovechar las columnas de texto mejoró más que cambiar de algoritmo. ¿Qué te dice eso sobre dónde conviene invertir el tiempo en un proyecto real?
- Llevamos cinco sesiones empezando la preparación con `select_dtypes(include=np.number)`. Era una decisión, no un trámite. ¿Qué otras decisiones de este tipo has ido tomando sin pensarlas?
- Has cerrado el bloque con un modelo un 35 % mejor que el primero y completamente opaco. Si el cliente te pregunta "¿por qué esta casa vale 300.000?", ¿qué le respondes? ¿Qué necesitarías para responderle bien? (Guarda la respuesta: es S14.)
- Si mañana empiezas un proyecto nuevo de regresión sobre datos tabulares, ¿por qué modelo empiezas y por qué? Da una respuesta corta y defendible.

---
## Cierre del bloque 2 — Regresión

Cinco sesiones, el mismo tipo de problema y un recorrido que conviene tener ordenado:

| Sesión | Modelo | Qué aportó | Qué costó |
|---|---|---|---|
| S03 | Regresión lineal | Coeficientes que se leen en euros | Solo relaciones rectas |
| S04 | Ridge y Lasso | Estabilidad y selección de variables | Hay que escalar; sigue siendo lineal |
| S05 | Árbol | No linealidad e interacciones, gratis | Inestable; no extrapola |
| S06 | Random Forest | Estabilidad por promediado | Ya no hay nada que dibujar |
| S06 | Boosting moderno | El mejor rendimiento, y rápido | Opaco y con hiperparámetros que importan |

**La regla práctica** para un problema tabular nuevo: empieza por un modelo lineal como referencia —es rápido y te dice si el problema es fácil—, pon al lado un Random Forest por defecto, y si la diferencia entre ambos es grande, invierte en boosting moderno. Si es pequeña, quédate con el lineal y su interpretabilidad.

**Lo que no hemos resuelto y viene después:**

- El modelo final es una caja negra. Explicar sus predicciones una a una es **S14**, con SHAP.
- Hemos usado el conjunto de test más veces de las que deberíamos, y en dos sesiones lo hemos hecho mal a propósito. Ordenarlo bien con pipelines es **S13**.
- Todo lo de hoy vale igual para clasificación, cambiando la métrica. Eso empieza en **S08**.